In [0]:
# ================================================================
# NOTEBOOK: nb_silver_orders_initial
# PURPOSE:  One-time full load from Bronze → Silver
# RUN:      ONCE only — never run again after first execution
# ================================================================


from pyspark.sql import functions as F
from pyspark.sql.functions import col, trim, when, upper, to_timestamp
from pyspark.sql.window import Window

BRONZE_PATH = "abfss://source@stshopsensedevhj.dfs.core.windows.net/bronze/orders/"
SILVER_PATH = "abfss://source@stshopsensedevhj.dfs.core.windows.net/silver/orders/"

# Read full Bronze
bronze_df = spark.read.parquet(BRONZE_PATH)
print(f"[BRONZE] Rows read: {bronze_df.count()}")


# Deduplication
dedup_window = Window.partitionBy("OrderID").orderBy(F.desc("LastModifiedDate"))
bronze_df = (
    bronze_df
    .withColumn("_rn", F.row_number().over(dedup_window))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

# Data Quality
bronze_df = (
    bronze_df
    .filter(col("OrderID").isNotNull())
    .filter(col("CustomerID").isNotNull())
    .filter(col("TotalAmount").isNotNull())
)

# Type casting, cleaning, derived columns
silver_df = (
    bronze_df
    .withColumn("OrderDate",        to_timestamp("OrderDate"))
    .withColumn("ShippedDate",      to_timestamp("ShippedDate"))
    .withColumn("DeliveredDate",    to_timestamp("DeliveredDate"))
    .withColumn("LastModifiedDate", to_timestamp("LastModifiedDate"))
    .withColumn("TotalAmount",      col("TotalAmount").cast("decimal(10,2)"))
    .withColumn("DiscountAmount",   col("DiscountAmount").cast("decimal(10,2)"))
    .withColumn("ShippingCharges",  col("ShippingCharges").cast("decimal(10,2)"))
    .withColumn("OrderStatus",      upper(trim(col("OrderStatus"))))
    .withColumn("PaymentMethod",    upper(trim(col("PaymentMethod"))))
    .withColumn("IsPrimeOrder",     col("IsPrimeOrder") == "TRUE")

    # Derived Columns
    .withColumn("OrderYear", F.year("OrderDate"))
    .withColumn("OrderMonth", F.month("OrderDate"))
    .withColumn("OrderDayOfWeek", F.dayofweek("OrderDate"))
    .withColumn("IsWeekendOrder",   col("OrderDayOfWeek").isin([1,7]))
    .withColumn("IsDelivered",      col("OrderStatus") == "DELIVERED")
    .withColumn("IsCancelled",      col("OrderStatus") == "CANCELLED")
    .withColumn("IsReturned",       col("OrderStatus") == "RETURNED")
    .withColumn("NetAmount",        col("TotalAmount") - col("DiscountAmount") - col("ShippingCharges"))
    .withColumn("DaysToDeliver",    F.when(col("DeliveredDate").isNotNull(),
                                           F.datediff(col("DeliveredDate"), col("OrderDate"))).otherwise(None))
    .withColumn("_silver_load_ts",  F.current_timestamp())
    .withColumn("_source",          F.lit("initial_full_load"))    
    .withColumn("_is_deleted",      F.lit(False))
)

    # Write to silver layer

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("OrderYear", "OrderMonth")
    .save(SILVER_PATH)

)

total = silver_df.filter(col("IsDelivered")).count()
delivered = silver_df.filter(col("IsDelivered")).count()
cancelled = silver_df.filter(col("IsCancelled")).count()

print(f"[OK]Silver orders written:{total} rows")
print(f"    Delivered:{delivered}| Cancelled: {cancelled}")
print(f"    Path:{SILVER_PATH}")
print("[DONE] Run nb_silver_orders_cdc_merge daily from now on.")







[BRONZE] Rows read: 3002
[OK]Silver orders written:1453 rows
    Delivered:1453| Cancelled: 405
    Path:abfss://source@stshopsensedevhj.dfs.core.windows.net/silver/orders/
[DONE] Run nb_silver_orders_cdc_merge daily from now on.


In [0]:
display(silver_df)

OrderID,CustomerID,SellerID,OrderDate,ShippedDate,DeliveredDate,OrderStatus,PaymentMethod,ShippingCity,ShippingState,ShippingPinCode,TotalAmount,DiscountAmount,ShippingCharges,IsPrimeOrder,LastModifiedDate,OrderYear,OrderMonth,OrderDayOfWeek,IsWeekendOrder,IsDelivered,IsCancelled,IsReturned,NetAmount,DaysToDeliver,_silver_load_ts,_source,_is_deleted
ORD0000001,CUST00092,SELL015,2024-03-19T10:40:27Z,2024-03-22T10:40:27Z,null,CANCELLED,CREDITCARD,Chennai,Tamil Nadu,239716,38040.90,125.58,0.00,true,2026-06-29T21:59:52.95Z,2024,3,3,false,false,true,false,37915.32,null,2026-07-08T14:35:24.018071Z,initial_full_load,false
ORD0000002,CUST00216,SELL047,2024-05-01T03:40:34Z,null,null,PROCESSING,CREDITCARD,Chennai,Tamil Nadu,931116,45118.57,186.99,0.00,true,2024-05-01T03:40:34Z,2024,5,4,false,false,false,false,44931.58,null,2026-07-08T14:35:24.018071Z,initial_full_load,false
ORD0000003,CUST00436,SELL037,2024-01-30T13:39:36Z,null,null,PROCESSING,NETBANKING,Bangalore,Karnataka,186995,47352.58,264.36,0.00,false,2024-01-30T13:39:36Z,2024,1,3,false,false,false,false,47088.22,null,2026-07-08T14:35:24.018071Z,initial_full_load,false
ORD0000004,CUST00286,SELL001,2024-03-02T10:25:54Z,2024-03-04T10:25:54Z,2026-07-03T19:29:51.91Z,RETURNED,WALLET,Mumbai,Maharashtra,313662,18253.07,466.57,71.12,true,2026-07-03T19:30:31.64Z,2024,3,7,true,false,false,true,17715.38,853,2026-07-08T14:35:24.018071Z,initial_full_load,false
ORD0000005,CUST00184,SELL009,2024-04-27T13:52:37Z,null,null,PROCESSING,CREDITCARD,Mumbai,Maharashtra,849883,8207.44,395.92,0.00,true,2024-04-27T13:52:37Z,2024,4,7,true,false,false,false,7811.52,null,2026-07-08T14:35:24.018071Z,initial_full_load,false
ORD0000006,CUST00088,SELL031,2024-04-07T17:03:38Z,2024-04-08T17:03:38Z,2024-04-09T17:03:38Z,RETURNED,WALLET,Mumbai,Maharashtra,185102,46845.79,316.01,77.04,false,2026-07-03T19:30:31.64Z,2024,4,1,true,false,false,true,46452.74,2,2026-07-08T14:35:24.018071Z,initial_full_load,false
ORD0000007,CUST00461,SELL004,2024-03-05T00:30:09Z,2024-03-07T00:30:09Z,2024-03-09T00:30:09Z,DELIVERED,CREDITCARD,Jaipur,Rajasthan,564490,13026.31,202.03,0.00,false,2024-03-05T00:30:09Z,2024,3,3,false,true,false,false,12824.28,4,2026-07-08T14:35:24.018071Z,initial_full_load,false
ORD0000008,CUST00068,SELL048,2024-05-14T22:55:46Z,null,null,PROCESSING,UPI,Mumbai,Maharashtra,737570,47071.93,62.18,57.76,true,2024-05-14T22:55:46Z,2024,5,3,false,false,false,false,46951.99,null,2026-07-08T14:35:24.018071Z,initial_full_load,false
ORD0000009,CUST00395,SELL032,2024-03-20T14:09:14Z,2024-03-23T14:09:14Z,2024-03-26T14:09:14Z,DELIVERED,CREDITCARD,Chennai,Tamil Nadu,987226,13027.41,428.93,0.00,true,2024-03-20T14:09:14Z,2024,3,4,false,true,false,false,12598.48,6,2026-07-08T14:35:24.018071Z,initial_full_load,false
ORD0000010,CUST00300,SELL004,2024-04-07T07:54:51Z,null,null,CANCELLED,NETBANKING,Mumbai,Maharashtra,326830,18972.32,168.38,0.00,false,2024-04-07T07:54:51Z,2024,4,1,true,false,true,false,18803.94,null,2026-07-08T14:35:24.018071Z,initial_full_load,false
